# Synthetic Benchmark Data: Generation and Exploration

This notebook is the **single source of truth for the synthetic benchmark corpora** used by Polarized Trees.

It has four jobs:

1. define the three benchmark regimes from the paper;
2. generate each corpus **once** with a fixed seed;
3. inspect the generated data and its ground truth;
4. save the datasets and ground truth so the benchmark notebook can reuse exactly the same data for every hyperparameter configuration.

The benchmark corpora are A (default), B (weak signal), and C (deep intersectional). A separate **unseen inference corpus** is also generated with the A/default regime and a new seed. The benchmark notebook uses only A/B/C for model selection and uses the unseen corpus only after model selection, matching the paper's train/validation → inference logic.

## 1. Imports and configuration

The generator is imported from `polartox.datagen`; the notebook does not redefine `AnnotatorPool`.

The benchmark settings below are the ones reported in the paper. In the final package, `AnnotatorPool` should expose the same generation interface used here, including `intensity_range` and `depth_weights`.

In [ ]:
# If running from a clean environment, install the local package once:
# %pip install -e ..


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Expected benchmark_config.py in the current "
        "directory or its parent."
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ------------------------------------------------------------
# Package imports
# ------------------------------------------------------------

from polartox.datagen import AnnotatorPool

from benchmark_config import (
    DIMS_DICT,
    DIMS,
    SCALE,
    N_TEXTS,
    ANNOTATORS_PER_IDENTITY,
    NOISE,
    CORPUS_CONFIGS,
    INFERENCE_CONFIG,
)


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "benchmark_data"

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_DIR = DATA_DIR


# ------------------------------------------------------------
# Configuration summary
# ------------------------------------------------------------

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

print("Dimensions:", DIMS)

print(
    "Benchmark corpora:",
    [c["name"] for c in CORPUS_CONFIGS],
)

print(
    "Inference corpus:",
    INFERENCE_CONFIG["name"],
)

## 2. Visualize the intended benchmark regimes

This reproduces the paper's benchmark-distribution view: latent polarization strength on the top row and interaction depth `k` on the bottom row.

These are the **sampling distributions specified by the benchmark**, not a second dataset-generation process. The actual generated corpora are inspected below.

In [ ]:
titles = ["A – Default", "B – Weak signal", "C – Deep"]
colors = ["tab:blue", "tab:orange", "tab:green"]
depths = [0, 1, 2, 3, 4]

fig, axes = plt.subplots(
    2, 3,
    figsize=(11, 5),
    constrained_layout=True,
)

for col, cfg in enumerate(CORPUS_CONFIGS):
    rng = np.random.default_rng(cfg["seed"])

    intensities = rng.uniform(
        cfg["intensity_range"][0],
        cfg["intensity_range"][1],
        size=N_TEXTS,
    )

    sampled_depths = rng.choice(
        depths,
        size=N_TEXTS,
        p=[cfg["depth_weights"][d] for d in depths],
    )

    # Top row: Latent polarization
    ax = axes[0, col]
    ax.hist(
        intensities,
        bins=10,
        range=(0, 1),
        density=True,
        color=colors[col],
        edgecolor="black",
        linewidth=0.8,
        alpha=0.85,
    )

    ax.set_xlim(0, 1)
    ax.set_title(titles[col], fontsize=11, fontweight="bold")
    ax.set_xlabel("Latent polarization")

    if col == 0:
        ax.set_ylabel("Density")

    ax.grid(axis="y", alpha=0.25)

    # Bottom row: Interaction depth
    ax = axes[1, col]

    counts = [
        np.sum(sampled_depths == d)
        for d in depths
    ]

    ax.bar(
        depths,
        counts,
        width=0.8,
        color=colors[col],
        edgecolor="black",
        linewidth=0.8,
    )

    ax.set_xticks(depths)
    ax.set_xlabel("Interaction depth")

    if col == 0:
        ax.set_ylabel("Texts")

    ax.grid(axis="y", alpha=0.25)

fig.suptitle(
    "Synthetic benchmark distributions",
    fontsize=14,
    fontweight="bold",
)

plt.show()

## 3. Generate the benchmark and inference corpora

Each corpus is generated **once**. The resulting datasets are reused unchanged by the benchmark notebook, so a hyperparameter comparison is not confounded by resampling the annotations.

Ground truth is retained only for synthetic recovery evaluation. The inference corpus is saved with its ground truth for optional post-hoc checking, but the benchmark notebook intentionally does **not** pass that ground truth to Polarized Trees during inference.

In [ ]:
def generate_corpus(cfg):
    pool = AnnotatorPool(
        dimensions=DIMS_DICT,
        scale=SCALE,
        intensity_range=cfg["intensity_range"],
        depth_weights=cfg["depth_weights"],
        annotators_per_identity=ANNOTATORS_PER_IDENTITY,
    )

    dataset, ground_truth = pool.generate_dataset(
        n_texts=N_TEXTS,
        n_annotators_per_text=None,
        noise=NOISE,
        seed=cfg["seed"],
    )

    return pool, dataset, ground_truth


def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


corpora = {}

for cfg in CORPUS_CONFIGS + [INFERENCE_CONFIG]:
    pool, dataset, ground_truth = generate_corpus(cfg)

    name = cfg["name"]
    corpora[name] = {
        "dataset": dataset,
        "ground_truth": ground_truth,
        "config": cfg,
        "pool_size": pool.pool_size,
        "n_identities": pool.n_identities,
    }

    dataset.to_csv(OUTPUT_DIR / f"{name}_dataset.csv", index=False)
    with open(OUTPUT_DIR / f"{name}_ground_truth.json", "w") as f:
        json.dump(json_safe(ground_truth), f, indent=2)

    with open(OUTPUT_DIR / f"{name}_config.json", "w") as f:
        json.dump(json_safe(cfg), f, indent=2)

    print(
        f"{name}: {len(dataset):,} annotations | "
        f"{dataset['text_id'].nunique()} texts | "
        f"{dataset['annotator_id'].nunique()} annotators"
    )

print("\nSaved corpora to:", OUTPUT_DIR.resolve())

## 4. Basic corpus inspection

Before running Polarized Trees, check that the generated datasets have the expected size, dimensions, annotation counts, and ground-truth depth distribution.

In [ ]:
for name, item in corpora.items():
    dataset = item["dataset"]
    gt = item["ground_truth"]

    k = pd.Series(
        {text_id: len(gt[text_id]["active_dims"]) for text_id in gt},
        name="k",
    )

    print(f"\n{name}")
    print("-" * len(name))
    print("Shape:", dataset.shape)
    print("Annotations/text:", dataset.groupby("text_id").size().describe().loc[["min", "mean", "max"]].to_dict())
    print("k distribution:")
    print(k.value_counts().sort_index().to_dict())

## 5. nDFU Exploration

We compute normalized DFU (nDFU) for each text to characterize the polarization induced by the three synthetic regimes. Each text is assigned an nDFU value from its annotation distribution and linked to its ground-truth number of active dimensions, \(k\).

We examine nDFU both **overall** and **by \(k\)**. The results are shown separately for A (default), B (weak signal), and C (deep), as well as in aggregated comparisons across the three corpora. This provides a descriptive check that the generated datasets exhibit the intended polarization structure before benchmarking Polarized Trees.

In [ ]:
# ndfu is installed as a core dependency of polartox.
from ndfu import dfu


In [ ]:
from ndfu import dfu

def ndfu_for_histogram(hist):
    hist = np.asarray(hist, dtype=float)
    total = hist.sum()
    if total == 0:
        return np.nan

    p = hist / total
    value = dfu(p)

    # Support either a scalar DFU-like return or a tuple/dict containing
    # the normalized score, depending on the installed ndFu version.
    if np.isscalar(value):
        return float(value)

    if isinstance(value, dict):
        for key in ("ndfu", "nDFU", "dfu"):
            if key in value:
                return float(value[key])

    if isinstance(value, (tuple, list, np.ndarray)):
        arr = np.asarray(value).ravel()
        if arr.size == 1:
            return float(arr[0])

    raise TypeError(
        "Could not interpret ndFu.dfu output. Inspect `dfu(...)` and "
        "adapt only this helper to the installed ndFu version."
    )


def overall_ndfu(dataset, scale):
    out = []

    for text_id, group in dataset.groupby("text_id", sort=True):
        counts = (
            group["rating"]
            .value_counts()
            .reindex(range(1, scale + 1), fill_value=0)
            .to_numpy()
        )
        out.append((text_id, ndfu_for_histogram(counts)))

    return pd.DataFrame(out, columns=["text_id", "ndfu"])


for name in corpora:
    nd = overall_ndfu(corpora[name]["dataset"], SCALE)

    gt = corpora[name]["ground_truth"]
    nd["k"] = nd["text_id"].map(
        lambda t: len(gt[int(t)]["active_dims"])
    )

    print(f"\n{name} nDFU summary")
    print(nd["ndfu"].describe())

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(nd["ndfu"].dropna(), bins=20)
    ax.set_xlabel("nDFU")
    ax.set_ylabel("Texts")
    ax.set_title(f"{name}: overall nDFU")
    plt.show()

    print("nDFU by k:")
    print(nd.groupby("k")["ndfu"].agg(["mean", "min", "max", "count"]))

    fig, ax = plt.subplots(figsize=(6, 4))
    nd.boxplot(column="ndfu", by="k", ax=ax)
    ax.set_title(f"{name}: nDFU by active dimensions (k)")
    ax.set_xlabel("k")
    ax.set_ylabel("nDFU")
    plt.suptitle("")
    plt.show()

In [ ]:
ndfu_data = {}

for name in corpora:
    nd = overall_ndfu(corpora[name]["dataset"], SCALE)
    ndfu_data[name] = nd["ndfu"].dropna().to_numpy()

In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green"]
labels = ["A – Default", "B – Weak signal", "C – Deep"]

fig, ax = plt.subplots(figsize=(7, 4.5))

for (name, values), color, label in zip(
    ndfu_data.items(), colors, labels
):
    ax.hist(
        values,
        bins=20,
        range=(0, 1),
        density=True,
        alpha=0.45,
        color=color,
        edgecolor="black",
        linewidth=0.7,
        label=label,
    )

ax.set_xlabel("nDFU")
ax.set_ylabel("Density")
ax.set_title("Overall nDFU distribution")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.show()

In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green"]
labels = ["A – Default", "B – Weak signal", "C – Deep"]

benchmark_names = [
    "A_default",
    "B_weak_signal",
    "C_deep",
]

ndfu_k_data = {}

for name in benchmark_names:
    nd = overall_ndfu(corpora[name]["dataset"], SCALE)

    gt = corpora[name]["ground_truth"]

    nd["k"] = nd["text_id"].map(
        lambda t: len(gt[int(t)]["active_dims"])
    )

    nd = nd.dropna(subset=["ndfu", "k"])
    nd["k"] = nd["k"].astype(int)

    ndfu_k_data[name] = nd

In [ ]:
ks = sorted(
    set().union(
        *[
            df["k"].unique()
            for df in ndfu_k_data.values()
        ]
    )
)

fig, ax = plt.subplots(figsize=(8, 4.5))

positions = []
box_data = []
box_colors = []

for i, k in enumerate(ks):
    for j, name in enumerate(benchmark_names):

        values = ndfu_k_data[name].loc[
            ndfu_k_data[name]["k"] == k,
            "ndfu"
        ]

        if len(values) == 0:
            continue

        positions.append(
            i + 1 + (j - 1) * 0.22
        )

        box_data.append(values)
        box_colors.append(colors[j])

bp = ax.boxplot(
    box_data,
    positions=positions,
    widths=0.18,
    patch_artist=True,
)

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.65)

ax.set_xticks(range(1, len(ks) + 1))
ax.set_xticklabels(ks)

ax.set_xlabel("Number of active dimensions (k)")
ax.set_ylabel("nDFU")
ax.set_title("nDFU by number of active dimensions")

handles = [
    plt.Line2D(
        [0], [0],
        color=color,
        marker="s",
        linestyle="",
        markersize=8,
        label=label,
    )
    for color, label in zip(colors, labels)
]

ax.legend(handles=handles)
ax.grid(axis="y", alpha=0.25)

plt.show()

## 6. Active-dimension balance

The synthetic benchmark should not systematically favor one demographic dimension. This check uses the stored ground truth and shows how often each dimension is active across each corpus.

In [ ]:
rows = []

for name in [c["name"] for c in CORPUS_CONFIGS]:
    gt = corpora[name]["ground_truth"]

    counts = {dim: 0 for dim in DIMS}
    total_texts = len(gt)

    for cfg in gt.values():
        for dim in cfg["active_dims"]:
            counts[dim] += 1

    for dim, count in counts.items():
        rows.append({
            "corpus": name,
            "dimension": dim,
            "active_texts": count,
            "frequency": count / total_texts,
        })

active_df = pd.DataFrame(rows)
display(active_df.pivot(index="dimension", columns="corpus", values="frequency"))

In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green"]
labels = ["A – Default", "B – Weak signal", "C – Deep"]

# Keep the corpus order fixed
active_pivot = active_df.pivot(
    index="dimension",
    columns="corpus",
    values="frequency",
).reindex(columns=[
    "A_default",
    "B_weak_signal",
    "C_deep",
])

fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(active_pivot.index))
width = 0.25

for i, (corpus, color, label) in enumerate(
    zip(active_pivot.columns, colors, labels)
):
    ax.bar(
        x + (i - 1) * width,
        active_pivot[corpus],
        width,
        color=color,
        edgecolor="black",
        linewidth=0.7,
        label=label,
    )

ax.set_xticks(x)
ax.set_xticklabels(active_pivot.index)
ax.set_ylabel("Proportion of texts")
ax.set_xlabel("Dimension")
ax.set_title("Frequency of active dimensions across synthetic corpora")

ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.show()

## 7. Ready for benchmark

At this point the synthetic environment has produced four fixed corpora:

- `A_default`, `B_weak_signal`, `C_deep`: **model-selection corpora**
- `inference_unseen`: **held-out inference corpus**

The next notebook reads these files, runs the randomized hyperparameter search on A/B/C, selects the configuration by mean Jaccard, and then runs the selected configuration on `inference_unseen` **without ground truth** to obtain F, C, P, and diagnostics.

In [ ]:
from pathlib import Path
import shutil

# ============================================================
# Package the generated synthetic benchmark data
# ============================================================

BUNDLE_DIR = PROJECT_ROOT / "synthetic_benchmark_bundle"
DATA_DIR = PROJECT_ROOT / "benchmark_data"

# Start from a clean bundle
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

BUNDLE_DATA_DIR = BUNDLE_DIR / "benchmark_data"
BUNDLE_DATA_DIR.mkdir(parents=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"{DATA_DIR} was not found."
    )

for path in DATA_DIR.iterdir():
    if path.is_file():
        shutil.copy2(path, BUNDLE_DATA_DIR / path.name)

print("Synthetic benchmark bundle:")
for path in sorted(BUNDLE_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(BUNDLE_DIR)}")

ZIP_PATH = shutil.make_archive(
    str(PROJECT_ROOT / "synthetic_benchmark_bundle"),
    "zip",
    root_dir=PROJECT_ROOT,
    base_dir=BUNDLE_DIR.name,
)

print(f"\nCreated: {ZIP_PATH}")
print(
    f"Size: {Path(ZIP_PATH).stat().st_size / 1024**2:.2f} MB"
)
print("The ZIP is in the project root.")
